In [57]:
# Import the required packages
import os
from os.path import dirname as one_folder_up, join as join_paths
import sqlite3

# Import pandas to read query outputs as dfs, read source files
import pandas as pd

from datetime import datetime

# For printing exceptions and logging
import traceback
import logging

In [58]:
# Define the functions

# START - Utils
def log_divider():
    return "-" * 100
# END - Utils

# START - DB
# Connects to an sqlite db and returns the conn object
def connect_to_database(db_path):
    try:
        conn = sqlite3.connect(db_path)
        return conn
    except Exception as e:
        return False

# Execute a select query against a DB
def execute_db_select(conn, query):
    logging.info(log_divider())
    try:
        logging.info(f"Running the SELECT query: {query}")
        df = pd.read_sql_query(query, conn)
        logging.info(f"SELECT successful! {len(df)} rows returned")
        return df
    except Exception as e:
        logging.warning(f"SELECT unsuccessful!")
        logging.error(traceback.print_exc())
        return False

# Gets an insert query along with a pandas dataframe, inserts the dataframe into the DB  
def execute_db_insert(conn, query, df):
    try:
        logging.info(f"INSERTing using the query: {query}")
        data_to_insert = df.to_records(index = False)
        # Create cursor, execute the query, commit, then close the cursor
        cur = conn.cursor()
        cur.executemany(query, data_to_insert)
        conn.commit()
        cur.close()
        logging.info(f"INSERT successful! {len(data)} rows inserted")
        return True
    except Exception as e:
        logging.warning(f"INSERT unsuccessful!")
        logging.error(traceback.print_exc())
        return False

# Deletes rows from a table based on an array of ids for a particular column
def execute_db_delete(conn, query, ids):
    try:
        logging.info(f"DELETEing using the query: {query}")
        logging.info(f"IDs to delete: {ids}")
        # Create cursor, execute the query, commit, then close the cursor
        cur = conn.cursor()
        cur.executemany(query, ids)
        conn.commit()
        cur.close()
        logging.info(f"DELETE successful! {len(ids)} rows deleted")
        return True
    except Exception as e:
        logging.warning(f"DELETE unsuccessful!")
        logging.error(traceback.print_exc())
        return False

In [59]:
# Define metadata variables
_JOB_START_DATETIME_    = datetime.now()
_JOB_START_DATETIME_STR_    = _JOB_START_DATETIME_.strftime("%Y_%m_%d_%H_%M_%S")
_JOB_NAME_              = "INGEST_CET_ENROLLMENTS"
_JOB_ID_                = f"{_JOB_NAME_}_{_JOB_START_DATETIME_STR_}"
_JOB_SOURCE_            = "CSV file - enrollments"
_JOB_DESTINATION_       = "SQLite DB - courses/enrollments"

In [60]:
# Define parameters
source_file             = r"enrolments.csv"
target_db               = "courses.db"
target_table            = "enrollments"
target_fields           = [
                            "enrollment_id",
                            "participant_id",
                            "participant_name",
                            "course_id",
                            "course_date",
                            "amount",
                            "subsidy",
                            "credits_used"
                        ]       

In [61]:
# Derive required variable(s)
current_folder          = os.getcwd()
root_folder             = one_folder_up(current_folder) 
""" 
    As the current file sits here "\gic_assignment\1_Data_Ingestion_Pipeline\"
    and the root folder is gic_assignment. 
"""

data_folder             = join_paths(root_folder, "data")
input_folder            = join_paths(data_folder, "input")
output_folder           = join_paths(data_folder, "output")

source_file_path        = join_paths(input_folder, source_file)
target_db_path          = join_paths(output_folder, target_db)

# Logging variables
log_folder              = join_paths(current_folder, "logs")
log_file                = f"{_JOB_ID_}.log"
log_file_path           = join_paths(log_folder, log_file)

In [62]:
# Init Logging
# Delete the log file if it already exists
if os.path.exists(log_file_path):
    os.remove(log_file_path)
FORMAT = '%(asctime)s: %(message)s'
logging.basicConfig(filename = log_file_path, format = FORMAT, level = logging.INFO)

# Log the important info so far
logging.info(f"Job started at {_JOB_START_DATETIME_STR_}")
logging.info(f"Job ID - {_JOB_ID_}, Job Name - {_JOB_NAME_}")
logging.info(f"Job Source - {_JOB_SOURCE_}, Job Destination - {_JOB_DESTINATION_}")
logging.info(log_divider())
logging.info("Context Parameters:")
logging.info(f"Root folder  - {root_folder}")
logging.info(f"Source file path - {source_file_path}")
logging.info(f"Target db path - {target_db_path}")
logging.info(f"Target table - {target_table}")

In [63]:
def main():
    logging.info(log_divider())

    # Create the target db conn
    conn = connect_to_database(target_db_path)
    logging.info(f"Target database connection created  - {conn}")
    
    # Read the input file as a pandas df
    enrollment_df = pd.read_csv(source_file_path)
    logging.info(f"Source file  has been read! {enrollment_df.shape[0]} rows found")

    # Delete rows in the target table if the enrolment IDs are already present
    ids_to_delete = list(enrollment_df['enrollment_id'])
    delete_query = f"DELETE FROM {target_table} WHERE enrollment_id = ?"
    query_status = execute_db_delete(conn, delete_query, ids_to_delete)

    # Check if the delete was successful
    if query_status:
        # Insert rows into the target table
        # Create the insert query using the list of 'target_fields'
        insert_query = f"INSERT INTO {target_table} ({', '.join(target_fields)}) VALUES ({', '.join('?' * len(target_fields))})"
        query_status = execute_db_insert(conn, insert_query, enrollment_df)

        # Check if the insert was successful
        if query_status:
            logging.info(f"Pipeline complete! Exiting...")
        else:
            # If the insert wasn't successful, close the db connection and exit
            conn.close()
            raise Exception("Encountered an error while inserting the new data! Exiting...")
    else:
        # If the delete wasn't successful, close the db connection and exit
        conn.close()
        raise Exception("Encountered an error while deleting the existing records! Exiting...")

# Call the main function
main()

Traceback (most recent call last):
  File "C:\Users\tatil\AppData\Local\Temp\ipykernel_1700\423586080.py", line 54, in execute_db_delete
    cur.executemany(query, ids)
sqlite3.ProgrammingError: parameters are of unsupported type


Exception: Encountered an error while deleting the existing records! Exiting...